# Multimodal Cancer Classification Challenge 2026 — v21 (v19 + ResNet-50 + 224)

**v19 LB 0.7455 = our new baseline.** v20 (paper-aligned overhaul) regressed to 0.5974 because heavy mixup + over-regularization prevented the model from fitting train in 12 epochs (train AUC capped at 0.75 vs v19's 0.99).

**v21 = v19's exact working recipe with two targeted paper-aligned changes**: backbone swap to ResNet-50 and input upscale to 224 (matches ImageNet pretrain). Everything else stays at v19's proven values.

## LB context

| # | Team | Score |
|---|---|---|
| 1 | Group 1 | 0.7832 (**+0.038 ahead**) |
| 2 | Group 2 | 0.7713 (+0.026 ahead) |
| **3** | **Rafael (you)** | **v19 LB 0.7455 ← OUR ANCHOR** |

## What changes vs v19, and why ONLY these two

v20 stacked 8+ changes from v19 and lost 0.15 LB. We can't afford another shotgun. v21 changes **exactly two** things, both supported by L1a slide 30 + Lian 2025 + Lu 2020:

| Lever | v19 (0.7455 ✓) | **v21** | Why this change |
|---|---|---|---|
| **Backbone** | EfficientNet-B0 (~5M each branch) | **ResNet-50 (~25M each branch)** | Teacher's own paper (Lu 2020) explicitly used ResNet-50 or DenseNet-201; Lian 2025 also ResNet-50. EffNet-B0 may be capacity-limited. |
| **Input resolution** | 128 native | **224 (bilinear upscaled)** | ResNet-50 ImageNet pretrained weights expect 224. At 128 the final feature map is 4×4 (16 cells) vs 7×7 (49 cells) at 224 — 3× more spatial signal. |

## What stays IDENTICAL to v19 (do not touch — these worked)

| Lever | v19 = v21 value | Why preserved |
|---|---|---|
| MIL aux loss | **ON, weight 0.5** | v19 proved this helps |
| Mixup | **OFF (α=0)** | v20's α=0.8 + 12 epochs = undertrained. Keep off. |
| Strong aug | **ON** (ColorJitter 0.4 + RandomErasing 0.25 + paired affine + D4 + ±10° rot) | v19 used this. Works. |
| Stain norm to test stats | **ON** | Cheap, no downside |
| AdaBN at inference | **ON** | Cheap, no downside |
| TTA | **8-way D4** (no multi-scale) | v19 used this. v20's 24-way may have added variance. |
| Dropout | **0.3** | v19 had 0.3, fit train AUC 0.99 cleanly. v20's 0.5 was over-regularized. |
| Label smoothing | **0.0** | v19 didn't use it. Don't add. |
| LR strategy | **Single LR 3e-4** for all params (no discriminative split) | v19 used this. v20's disc LR didn't help. |
| Weight decay | 1e-4 | v19 |
| **Validation tracking** | **OFF — all 12 patients in train** | **v20 held 2 patients out → lost 17% of training data + LB regressed by 0.15.** Held-out *train* patients are not OOD enough to predict Kaggle test patients. |
| Epochs | **12** | v19 |
| Effective batch | 128 (64 × grad_accum 2 — ResNet-50@224 doesn't fit batch 128 on T4) | matches v19's effective batch |

## Compute budget on T4

| Stage | Time |
|---|---|
| JPEG cache (one-time) | ~70 min |
| Test pixel stats | ~30 s |
| Train ResNet-50 @ 224 (12 ep × ~14 min/ep, all 12 patients) | ~170 min |
| 8-way D4 TTA + AdaBN @ 224 | ~12 min |
| **Total** | **~4h 15min** |

## Reading the v21 LB

- **LB ≥ 0.77**: ResNet-50 + 224 is the right architecture. v22 layers ONE more change (light mixup α=0.1 OR 18 epochs).
- **LB 0.74–0.77**: Architecture is fine but no big lift. v22 = ensemble of v21 + v19.
- **LB 0.70–0.74**: ResNet-50 added some capacity but not enough. v22 = swap to DenseNet-201.
- **LB < 0.70**: ResNet-50 + 224 hurts. v22 = revert to v19 + 18 epochs.

## IMPORTANT — DO NOT JUST CLICK RUN ALL

1. **Save Version** (green button top-right)
2. **Save & Run All (Commit)**
3. Description: `"v21: v19 recipe + ResNet-50 + 224 (only 2 changes from v19 0.7455)"`
4. Wait ~4h 15min
5. Submit `submission.csv`. Compare to v19 LB 0.7455.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === v21: only TWO changes vs v19 (0.7455 LB baseline) ===
USE_RESNET50        = True   # NEW vs v19: ResNet-50 backbone (was EfficientNet-B0)
USE_EFFICIENTNET    = False  # v19 had this True; turned off in favor of ResNet-50
INPUT_SIZE          = 224    # NEW vs v19: upscale 128->224 to match ImageNet pretrain

# === EVERYTHING BELOW IS IDENTICAL TO v19 (0.7455 baseline — do not touch) ===
USE_MIL_LOSS        = True   # v19 had this — KEEP
MIL_WEIGHT          = 0.5
USE_STRONG_AUG      = True
RANDOM_ERASING_P    = 0.25

USE_TEST_STAIN_NORM = True
USE_ADABN           = True
USE_MULTISCALE_TTA  = False  # 8-way D4 only — v19 used this
TTA_SCALES          = (192, 224, 256)  # only used if USE_MULTISCALE_TTA=True

LABEL_SMOOTHING     = 0.0    # v19 had this OFF — KEEP

# === Training (v19 settings, with grad accum to fit ResNet-50@224 on T4) ===
BASE_SEED   = 1
EPOCHS      = 12
BATCH_SIZE  = 64             # v19 used 128, but ResNet-50@224 needs less VRAM headroom on T4
GRAD_ACCUM  = 2              # effective batch = 64 * 2 = 128 (matches v19)
LR          = 3e-4           # v19 single LR — KEEP (no discriminative LR; that hurt in v20)
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.0            # v19 had this OFF — KEEP (v20's α=0.8 caused undertraining)
DROPOUT     = 0.3            # v19 had 0.3 — KEEP (v20's 0.5 over-regularized)

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4       # batch 64 / 4 = 16 cells per patient per batch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# v11 hardcoded stats (used when USE_TEST_STAIN_NORM=False)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

assert not (USE_RESNET50 and USE_EFFICIENTNET), "Pick at most one of RESNET50 / EFFICIENTNET"
_backbone_name = ("ResNet-50" if USE_RESNET50
                  else "EfficientNet-B0" if USE_EFFICIENTNET
                  else "ResNet-18")

print(f"\nConfig (v21 — v19 recipe + ResNet-50 + 224):")
print(f"  Backbone            = {_backbone_name}  (was EfficientNet-B0 in v19)")
print(f"  INPUT_SIZE          = {INPUT_SIZE}  (was 128 in v19; matches ImageNet pretrain)")
print(f"  BATCH_SIZE          = {BATCH_SIZE} * GRAD_ACCUM={GRAD_ACCUM} (effective {BATCH_SIZE*GRAD_ACCUM})")
print(f"  USE_MIL_LOSS        = {USE_MIL_LOSS}  weight={MIL_WEIGHT}  (v19 baseline; do not change)")
print(f"  MIXUP_ALPHA         = {MIXUP_ALPHA}  (v19 baseline — off)")
print(f"  DROPOUT             = {DROPOUT}  (v19 baseline)")
print(f"  LABEL_SMOOTHING     = {LABEL_SMOOTHING}  (v19 baseline)")
print(f"  LR                  = {LR}  (single LR — v19 baseline)")
print(f"  USE_STRONG_AUG      = {USE_STRONG_AUG}  erasing_p={RANDOM_ERASING_P}")
print(f"  USE_TEST_STAIN_NORM = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN           = {USE_ADABN}")
print(f"  USE_MULTISCALE_TTA  = {USE_MULTISCALE_TTA}  (v19 used 8-way D4 only)")
print(f"  EPOCHS              = {EPOCHS}")
print(f"  NO val split — all 12 patients in training (v19 baseline)")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v19: also returns patient_id for MIL grouping in training."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        # Patient ID is -1 for test rows (Name has no pat_X prefix in some splits but in this
        # dataset all names are pat_NN_image_MM.jpg so parse always succeeds).
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_resnet50_branch(pretrained=True):
    """v21 NEW: ResNet-50 dual-branch builder (per L1a slide 30 + Lian 2025 + Lu 2020)."""
    weights = "DEFAULT" if pretrained else None
    net = models.resnet50(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features  # 2048 for ResNet-50
    net.fc = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

def _make_branch(pretrained=True):
    if USE_RESNET50:
        return _make_resnet50_branch(pretrained)
    if USE_EFFICIENTNET:
        return _make_effnet_b0_branch(pretrained)
    return _make_resnet18_branch(pretrained)

class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_branch(pretrained)
        self.fl_branch, _  = _make_branch(pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        self._feat_dim = fd

    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    n_params = sum(p.numel() for p in _m.parameters())
    n_backbone = (sum(p.numel() for p in _m.bf_branch.parameters()) +
                  sum(p.numel() for p in _m.fl_branch.parameters()))
    n_head = sum(p.numel() for p in _m.head.parameters())
    print(f"Output shape: {_m(_x, _x).shape}")
    print(f"Params:       {n_params/1e6:.1f}M total = {n_backbone/1e6:.1f}M backbone + {n_head/1e6:.1f}M head")
    print(f"Backbone:     {_backbone_name}  (feature dim {_m._feat_dim})")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# Compute pixel statistics for stain normalization.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

In [ ]:
def _to_tensor_norm(mean, std):
    """v21: ToTensor -> Resize(INPUT_SIZE) -> Normalize.
    Resize to INPUT_SIZE (default 224) to match ImageNet pretrained backbone.
    Bilinear upscale from 128 is cheap.
    """
    def fn(img):
        t = TF.to_tensor(img)  # [1, 128, 128]
        if INPUT_SIZE != t.shape[-1]:
            t = TF.resize(t, [INPUT_SIZE, INPUT_SIZE], antialias=True)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    """D4 + small rotation, applied identically to BF and FL (paired)."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0,
                 affine_deg=0.0, affine_translate=0.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
        self.affine_deg = affine_deg; self.affine_translate = affine_translate
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        # v19: paired affine — same translate applied to both modalities (keeps BF/FL aligned)
        if self.affine_deg > 0 or self.affine_translate > 0:
            H, W = bf.shape[-2], bf.shape[-1]
            angle = random.uniform(-self.affine_deg, self.affine_deg) if self.affine_deg > 0 else 0.0
            tx = random.uniform(-self.affine_translate, self.affine_translate) * W if self.affine_translate > 0 else 0
            ty = random.uniform(-self.affine_translate, self.affine_translate) * H if self.affine_translate > 0 else 0
            bf = TF.affine(bf, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
            fl = TF.affine(fl, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
        return bf, fl

def train_modality_transform(modality):
    """v19/v21: stronger color jitter, with optional RandomErasing after normalization."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    if USE_STRONG_AUG:
        steps = [T.ColorJitter(brightness=0.4, contrast=0.4), norm]
        if RANDOM_ERASING_P > 0:
            steps.append(T.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.20),
                                         ratio=(0.3, 3.3), value=0.0))
        return T.Compose(steps)
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def build_paired_aug():
    """v19/v21 paired aug: D4 + ±10° rot, plus ±15° affine and 10% translate."""
    if USE_STRONG_AUG:
        return PairedGeoAug(max_rot=10.0, affine_deg=15.0, affine_translate=0.10)
    return PairedGeoAug(max_rot=10.0)

print(f"Augmentation summary:")
print(f"  Input resize:     128 -> {INPUT_SIZE}  (v21 NEW; matches ImageNet pretrain)")
print(f"  ColorJitter:      {'brightness/contrast=0.4' if USE_STRONG_AUG else 'brightness/contrast=0.2'}")
print(f"  RandomErasing:    p={RANDOM_ERASING_P if USE_STRONG_AUG else 0.0}")
print(f"  PairedGeoAug:     D4 + ±10° rot" + (" + ±15° affine + 10% translate" if USE_STRONG_AUG else ""))

In [ ]:
def smooth(y, eps):
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def mil_patient_loss(logits, y, patient_ids, pos_weight=None):
    """v19 per-patient mean-logit BCE loss — kept in v21."""
    unique_pids = torch.unique(patient_ids)
    if len(unique_pids) < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    p_logits, p_labels = [], []
    for pid in unique_pids:
        mask = patient_ids == pid
        p_logits.append(logits[mask].mean())
        p_labels.append(y[mask][0])
    p_logits = torch.stack(p_logits)
    p_labels = torch.stack(p_labels)
    return F.binary_cross_entropy_with_logits(p_logits, p_labels.float(),
                                              pos_weight=pos_weight)

def run_epoch_train(model, loader, optimizer, scaler, criterion_cell, sched,
                    pos_weight=None, grad_accum=1, log_every=200):
    """v21: v19 train loop + gradient accumulation support (for ResNet-50@224 on T4)."""
    model.train()
    losses, hard_ys, ps = [], [], []
    cell_losses, mil_losses = [], []
    t_last = time.time()
    optimizer.zero_grad(set_to_none=True)
    accum_count = 0
    for i, batch in enumerate(loader):
        bf  = batch["bf"].to(DEVICE, non_blocking=True)
        fl  = batch["fl"].to(DEVICE, non_blocking=True)
        y   = batch["label"].float().to(DEVICE, non_blocking=True)
        pid = batch["patient_id"].to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        # v19/v21: no mixup. y stays hard 0/1.
        y_s = smooth(y, LABEL_SMOOTHING)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss_cell = criterion_cell(logits, y_s)
            if USE_MIL_LOSS:
                loss_mil = mil_patient_loss(logits, y, pid, pos_weight=pos_weight)
                loss = loss_cell + MIL_WEIGHT * loss_mil
            else:
                loss_mil = torch.zeros((), device=logits.device)
                loss = loss_cell
            loss_scaled = loss / grad_accum

        if scaler is not None:
            scaler.scale(loss_scaled).backward()
        else:
            loss_scaled.backward()
        accum_count += 1
        if accum_count >= grad_accum:
            if scaler is not None:
                if GRAD_CLIP > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale: sched.step()
            else:
                if GRAD_CLIP > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step(); sched.step()
            optimizer.zero_grad(set_to_none=True)
            accum_count = 0

        losses.append(loss.item())
        cell_losses.append(loss_cell.item())
        mil_losses.append(loss_mil.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | "
                  f"total {float(np.mean(losses[-log_every:])):.4f} "
                  f"cell {float(np.mean(cell_losses[-log_every:])):.4f} "
                  f"mil {float(np.mean(mil_losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), float(np.mean(cell_losses)), float(np.mean(mil_losses)), auc

# === Train ONE full-data model (v19 baseline: all 12 patients, no val split) ===
print(f"\n=== v21: Training {_backbone_name} @ {INPUT_SIZE}x{INPUT_SIZE} "
      f"({EPOCHS} epochs, all {df_train['patient_id'].nunique()} patients) ===")
seed_everything(BASE_SEED + 100)
train_ds = CachedCellDataset(df_train, bf_train_cache, fl_train_cache,
                             train_modality_transform("bf"),
                             train_modality_transform("fl"),
                             paired_tf=build_paired_aug())
sampler = PatientBalancedSampler(df_train, batch_size=BATCH_SIZE,
                                 patients_per_batch=PATIENTS_PER_BATCH, seed=BASE_SEED + 100)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
pos = (df_train["Diagnosis"] == 1).sum()
neg = (df_train["Diagnosis"] == 0).sum()
pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
print(f"  pos_weight={pos_weight.item():.3f}  LR={LR}  MIL_W={MIL_WEIGHT if USE_MIL_LOSS else 0.0}  "
      f"backbone={_backbone_name}")
criterion_cell = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# OneCycleLR step count must match the number of optimizer.step() calls (=raw_steps / grad_accum)
steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None
print(f"  steps/epoch={len(train_loader)} (raw) / {steps_per_epoch} (after grad_accum={GRAD_ACCUM})")

history = []
for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_cell, tr_mil, tr_auc = run_epoch_train(
        model, train_loader, optimizer, scaler, criterion_cell, sched,
        pos_weight=pos_weight, grad_accum=GRAD_ACCUM)
    dt = time.time() - t0
    print(f"  ep {ep:>2d} | total {tr_loss:.4f} cell {tr_cell:.4f} mil {tr_mil:.4f} "
          f"tr_auc {tr_auc:.4f} | {dt:.1f}s")
    history.append({"epoch": ep, "tr_loss": tr_loss, "tr_cell": tr_cell,
                    "tr_mil": tr_mil, "tr_auc": tr_auc, "time": dt})

ckpt_path = OUT_DIR / "fulldata_best.pt"
torch.save({"model": model.state_dict(), "epoch": EPOCHS - 1,
            "args": {"dropout": DROPOUT, "backbone": _backbone_name,
                     "input_size": INPUT_SIZE}},
           ckpt_path)
with open(OUT_DIR / "history.json", "w") as f:
    json.dump({"history": history}, f, indent=2)
print(f"\nSaved {ckpt_path}")
del model, optimizer, sched, scaler, train_loader
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
epochs = [e["epoch"] for e in history]
ax[0].plot(epochs, [e["tr_loss"] for e in history], marker="o", color="tab:blue", label="total")
ax[0].plot(epochs, [e["tr_cell"] for e in history], marker="s", color="tab:purple", label="cell BCE")
ax[0].plot(epochs, [e["tr_mil"]  for e in history], marker="^", color="tab:orange", label="MIL patient BCE")
ax[0].legend(); ax[0].set(title="Train losses", xlabel="epoch", ylabel="loss")
ax[1].plot(epochs, [e["tr_auc"]  for e in history], marker="o", color="tab:green")
ax[1].set(title="Train AUC (cell-level)", xlabel="epoch", ylabel="AUC")
ax[2].plot(epochs, [e["time"]    for e in history], marker="o", color="tab:red")
ax[2].set(title="Epoch time (s)",  xlabel="epoch", ylabel="seconds")
for a in ax: a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# === AdaBN: update BN running stats on test data before inference ===
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict(ckpt_path, loader, tta_scales=None):
    model = load_model(ckpt_path)
    if USE_ADABN:
        print("  running AdaBN pass...")
        adabn_pass(model, loader)
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

# v21: at INPUT_SIZE=224 with ResNet-50, use smaller inference batch to stay within T4 VRAM
INFER_BATCH = 128 if INPUT_SIZE >= 192 else 256
test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=INFER_BATCH, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Predicting with {n_aug_total}-way TTA "
      f"(scales={scales_to_use or 'native'}, AdaBN={USE_ADABN}, batch={INFER_BATCH})")

t0 = time.time()
preds = predict(ckpt_path, test_loader, tta_scales=scales_to_use)
print(f"  done in {time.time()-t0:.1f}s")

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")
print(sub.head())
!wc -l /kaggle/working/submission.csv